# Registry defaults and overrides

Use `FinstackConfig.extensions` to override versioned registry defaults explicitly. The embedded registries remain the fallback; overrides are opt-in and travel with the config object.


## Setup

This notebook uses the local registry JSON files as starting payloads, edits copies, and stores the edited payloads under the same extension keys Rust uses when loading defaults from `FinstackConfig`.


In [ ]:
import json
from pathlib import Path
from pprint import pprint

from finstack_quant.core.config import FinstackConfig


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "Cargo.toml").exists() and (candidate / "finstack-quant/core/src").exists():
            return candidate
    raise RuntimeError("could not locate finstack-quant repository root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
REPO_ROOT


## Extension keys

Extension keys are versioned and namespaced. Use the exact key for the registry you want to replace. The margin registry keeps the legacy-compatible `valuations.margin_registry.v1` key, but the registry data is owned by `finstack-quant-margin`.


In [ ]:
import json
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/registry_defaults_and_overrides.json").read_text())

EXTENSION_KEYS = _NOTEBOOK_DATA['EXTENSION_KEYS']

for name, key in EXTENSION_KEYS.items():
    print(f"{name:34s} {key}")


## Override Monte Carlo defaults

Registry overrides are full payload replacements. Start from the embedded JSON, edit the fields you govern, then attach the full edited payload to the config.


In [ ]:
mc_defaults_path = REPO_ROOT / "finstack-quant/monte_carlo/data/defaults/pricer_defaults.v1.json"
mc_defaults = json.loads(mc_defaults_path.read_text())

print("Embedded Python European-pricer defaults:")
pprint(mc_defaults["python_bindings"]["european_pricer"])
print("Embedded default currency:", mc_defaults["python_bindings"]["default_currency"])


In [ ]:
mc_override = json.loads(json.dumps(mc_defaults))
mc_override["python_bindings"]["european_pricer"]["num_paths"] = 25_000
mc_override["python_bindings"]["european_pricer"]["seed"] = 20260426
mc_override["python_bindings"]["default_currency"] = "EUR"

# FinstackConfig() is the canonical way to obtain a default config (no .default() classmethod).
cfg = FinstackConfig()
cfg.set_extension(EXTENSION_KEYS["monte_carlo_defaults"], mc_override)

print("Configured extension keys:")
print(cfg.extension_keys())
print()
print("Overridden Python European-pricer defaults:")
pprint(cfg.get_extension(EXTENSION_KEYS["monte_carlo_defaults"])["python_bindings"]["european_pricer"])


## Add analytics calendar defaults

The fiscal calendar defaults used by analytics can be governed the same way. Here we keep the exchange calendar explicit because the start day needs business-day alignment logic.


In [ ]:
import json
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/registry_defaults_and_overrides.json").read_text())

analytics_defaults = _NOTEBOOK_DATA['analytics_defaults']

analytics_override = json.loads(json.dumps(analytics_defaults))
fiscal_calendar = analytics_override["python_bindings"]["lookback"]["default_fiscal_calendar"]
fiscal_calendar.update(
    {
        "id": "company_fiscal_april_nyse",
        "calendar_id": "nyse",
        "start_month": 4,
        "start_day": 1,
    }
)

cfg.set_extension(EXTENSION_KEYS["analytics_defaults"], analytics_override)

pprint(cfg.get_extension(EXTENSION_KEYS["analytics_defaults"])["python_bindings"]["lookback"])


## Serialize and round-trip

`FinstackConfig` can be serialized to JSON and restored. Use this payload when passing config into Python APIs that accept a config JSON string, or when handing config to Rust service code.


In [ ]:
config_json = cfg.to_json()
print(config_json[:500] + "...")

round_tripped = FinstackConfig.from_json(config_json)
print(round_tripped.extension_keys())
print(round_tripped.get_extension(EXTENSION_KEYS["monte_carlo_defaults"])["python_bindings"]["default_currency"])


## Remove an override

Because overrides are explicit, cleanup is just as explicit. Removing a key restores the embedded fallback for code that loads that registry from the config.


In [ ]:
removed = cfg.remove_extension(EXTENSION_KEYS["analytics_defaults"])
print("removed analytics override:", removed)
print("remaining keys:", cfg.extension_keys())


## Practical rule

Keep overrides as reviewed JSON artifacts. Load the embedded registry payload, change only the governed fields, validate in CI, and attach the whole payload to `FinstackConfig`. Avoid partial fragments unless a specific registry documents overlay semantics; margin is the existing overlay-style exception.


## Rating scale registry

The rating-scales registry is a user-facing `core` surface: use the embedded registry by default, or derive one from `FinstackConfig` extensions when an application overrides scales.

In [ ]:
from finstack_quant.core.config import FinstackConfig
from finstack_quant.core.rating_scales import (
    RATING_SCALES_EXTENSION_KEY,
    UnknownScalePolicy,
    embedded_registry,
    registry_from_config,
)

registry = embedded_registry()
configured = registry_from_config(FinstackConfig())
print("extension key:", RATING_SCALES_EXTENSION_KEY)
print("default scale:", registry.default_scale_id())
print("SP scale known:", registry.is_known_rating_scale("sp"))
print("from config default score:", configured.default_scorecard_score())
print("unknown policy:", UnknownScalePolicy.from_name("error"))